# DATA 5310 HW 8: Cyber Threats

**Group 2:** Maya Silver, Gurpreet Kaur, Badamgarav Battushig, Jack Lichwa

## Introduction

### Topic and Why It Matters

Cyber attacks are one of the biggest modern global issues. They hit governments, hospitals, banks, schools, and everyday businesses, and they are growing every year.

Our goal is to tell a data story about where attacks happen, who is behind them, which sectors are targeted, and which countries are hit hardest once we account for their size.

### Our Datasets and How They Connect

| # | Dataset | Role | Investigated by | What one row is |
|---|---|---|---|---|
| 1 | EuRepoC Global Dataset of Cyber Incidents | Core | Maya Silver | One cyber incident |
| 2 | Ransomware.live Victims | Supporting | Gurpreet Kaur | One ransomware victim claim |
| 3 | World Bank Country Indicators | Supporting | Badamgarav Battushig | One country in one year |
| 4 | [Name] | Supporting | Jack Lichwa | [What one row is] |

Datasets 1-3 share **country** (as a 2-letter code, e.g. US) and **year**, so they can be joined. All data files are stored in our group GitHub repository: https://github.com/SU-DataViz-Group-Fall-2026/ransomware-analysis

Each dataset section follows the same steps: data source, loading, describing, cleaning, exploratory data analysis (EDA), and summary. Run the cells in order from top to bottom, because later cells use variables from earlier ones.

In [ ]:
# install the two extra libraries we use (only needed once)
%pip install -q pycountry country_converter

In [ ]:
import json
import urllib.request

import pandas as pd
import altair as alt
import pycountry
import country_converter as coco

# allow charts with more than 5,000 rows and show more columns in tables
alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

# all data files are stored in our group GitHub repository
github_url = 'https://raw.githubusercontent.com/SU-DataViz-Group-Fall-2026/ransomware-analysis/main/data/'

---
## Dataset 1: EuRepoC Global Dataset of Cyber Incidents (Core)
*Investigated by Maya Silver*

### Step 1.1: Data Source

European Repository of Cyber Incidents (EuRepoC), *Global Dataset of Cyber Incidents*, version 1.3, Zenodo, March 2025. https://zenodo.org/records/14965395

EuRepoC is a research project that tracks publicly reported cyber incidents with a political dimension, such as attacks by state-backed groups or attacks on governments and critical infrastructure. Despite the name, it covers incidents worldwide, with start dates from 2000 through 2024. We downloaded the dataset from Zenodo and stored it in our group GitHub repository.

### Step 1.2: Loading the Data

In [ ]:
# load the EuRepoC dataset from our GitHub repository
eurepoc_raw = pd.read_csv(github_url + 'eurepoc_global_dataset_1_3.csv')
eurepoc_raw.head()

In [ ]:
eurepoc_raw.columns

In [ ]:
eurepoc_raw.shape

### Step 1.3: Describing the Data

#### What One Row Represents
One row is one cyber incident. The file has 3,414 rows and 85 columns.

#### Columns Overview
The columns fall into a few groups: basic details (ID, name, description, start and end dates), the incident type (data theft, disruption, hijacking, ransomware), the target (receiver name, country, region, and sector), the attacker (initiator name, country, and category, such as state or non-state), severity and impact scores (weighted intensity, impact indicator, economic impact), political and legal responses, and technical details like zero-day use and MITRE ATT&CK initial access and impact techniques.

#### Challenges and Limitations
- Many cells hold several values separated by semicolons. For example, an incident can have two incident types or several target countries, so these have to be split before counting.
- Missing data is mostly stored as text like "Not available", "Unknown", or "Not attributed" instead of true nulls. That's why `info()` shows almost no missing values even though a lot is missing in practice.
- Dates are stored as text in day.month.year format and need to be converted.
- The data only includes publicly reported incidents, so it reflects what gets reported and attributed, not everything that happens. Attribution is also uncertain. For incidents targeting the US, the two biggest attacker categories are "Not attributed" and "Unknown".

### Step 1.4: Cleaning the Data

#### Selecting and Renaming Columns
We kept 45 of the 85 columns, focusing on the incident type, target, attacker, dates, severity, impact, political response, and MITRE fields. The column names are already short and readable (lowercase with underscores), so they did not need renaming.

In [ ]:
cyber = eurepoc_raw[['incident_id', 'name', 'description',
            'incident_type', 'inclusion_criterion', 'source_disclosure',
            'receiver_name', 'receiver_country', 'receiver_regions',
            'receiver_category', 'receiver_subcategory',
            'initiator_name', 'initiator_country',
            'initiator_category', 'initiator_subcategory', 'settled_initiator',
            'start_date', 'end_date', 'attribution_date', 'political_response_date',
            'unweighted_intensity', 'target_multiplier', 'weighted_intensity',
            'impact_indicator_label', 'impact_indicator_score',
            'economic_impact', 'economic_impact_value', 'economic_impact_currency',
            'affected_entities', 'intelligence_impact', 'functional_impact',
            'number_political_responses', 'political_response_type',
            'political_response_responding_country', 'political_response_responding_actor',
            'zero_days', 'has_disruption', 'data_theft', 'disruption', 'hijacking',
            'physical_effects_spatial', 'physical_effects_temporal', 'user_interaction',
            'mitre_initial_access', 'mitre_impact']].copy()

cyber.head()

In [ ]:
cyber.info()

#### Handling Missing Values
Because missing values are stored as text, we count how often each placeholder appears in the key columns. "Not attributed" and "Unknown" are meaningful for this topic (many attacks are never traced back to an attacker), so we keep them as their own category instead of deleting those incidents. Incidents with no valid start date are removed in the next step, since they can't be placed on a timeline.

In [ ]:
# count placeholder text in the key columns
placeholders = ['Not available', 'Unknown', 'Not attributed']
for col in ['incident_type', 'receiver_country', 'receiver_category', 'initiator_country', 'start_date']:
    count = cyber[col].isin(placeholders).sum()
    print(f'{col}: {count} placeholder values')

#### Recoding Values
One victim category has a very long label, so we shorten it for charts.

In [ ]:
# shorten the long "Corporate Targets" label (it can appear inside multi-value cells)
cyber['receiver_category'] = cyber['receiver_category'].str.replace(
    'Corporate Targets (corporate targets only coded if the respective company is not part of the critical infrastructure definition)',
    'Corporate Targets', regex=False)

#### Transforming the Data
We convert the start date from text (day.month.year) to a real date and the intensity and impact scores to numbers, then remove incidents with no valid start date.

In [ ]:
# convert dates and scores to the right types
cyber['start_date'] = pd.to_datetime(cyber['start_date'], dayfirst=True, errors='coerce')
cyber['weighted_intensity'] = pd.to_numeric(cyber['weighted_intensity'], errors='coerce')
cyber['impact_indicator_score'] = pd.to_numeric(cyber['impact_indicator_score'], errors='coerce')

# remove incidents with no valid start date
print(cyber['start_date'].isna().sum(), 'incidents have no valid start date')
cyber = cyber.dropna(subset=['start_date'])
print(len(cyber), 'incidents kept')

**Cleaning recap:** kept 45 useful columns, checked placeholder values and kept "Not attributed" and "Unknown" as their own category, shortened a long category label, converted dates and scores to the right types, and removed incidents with no valid start date. After cleaning, each row is still one cyber incident.

### Step 1.5: Exploratory Data Analysis

For the exploration, we filtered to incidents where the United States is one of the target countries. We also wrote a small helper that splits the multi-value cells and counts each value separately, so an incident with two types is counted once for each type.

In [ ]:
# keep incidents where the US is one of the targets
us = cyber[cyber['receiver_country'].str.contains('United States', na=False)].copy()
print(len(us), 'incidents with the US as a target')

# helper: split cells with several values (a; b; c) and count each value
def count_values(data, col):
    out = (data[col].str.split(';').explode().str.strip()
           .value_counts().reset_index())
    out.columns = [col, 'incidents']
    return out

In [ ]:
types = count_values(us, 'incident_type')

alt.Chart(types).mark_bar().encode(
    alt.X('incidents:Q', title='Number of incidents'),
    alt.Y('incident_type:N', sort='-x', title='Incident type')
).properties(title='Types of Cyber Incidents Targeting the United States', width=450)

In [ ]:
by_year = us.groupby(us['start_date'].dt.year).size().reset_index(name='incidents')
by_year.columns = ['year', 'incidents']

alt.Chart(by_year).mark_bar().encode(
    alt.X('year:O', title='Start year'),
    alt.Y('incidents:Q', title='Number of incidents')
).properties(title='Cyber Incidents Targeting the United States by Year', width=500)

In [ ]:
attackers = count_values(us, 'initiator_country').head(10)

alt.Chart(attackers).mark_bar().encode(
    alt.X('incidents:Q', title='Number of incidents'),
    alt.Y('initiator_country:N', sort='-x', title='Attacker country')
).properties(title='Top Attacker Countries for Incidents Targeting the United States', width=450)

In [ ]:
sectors = (us['receiver_category'].str.split(';')
           .apply(lambda x: list(set(s.strip() for s in x)))
           .explode().value_counts().reset_index())
sectors.columns = ['receiver_category', 'incidents']

alt.Chart(sectors).mark_bar().encode(
    alt.X('incidents:Q', title='Number of incidents'),
    alt.Y('receiver_category:N', sort='-x', title='Target sector')
).properties(title='US Sectors Targeted in Cyber Incidents', width=450)

In [ ]:
# add a year column for the US incidents
us['year'] = us['start_date'].dt.year

In [ ]:
alt.Chart(us).mark_bar().encode(
    alt.X('weighted_intensity:Q', bin=alt.Bin(step=1), title='Weighted cyber intensity'),
    alt.Y('count():Q', title='Number of incidents')
).properties(title='How Severe Are Cyber Incidents Targeting the US?', width=450)

In [ ]:
alt.Chart(us).mark_circle(opacity=0.4, size=50).encode(
    alt.X('start_date:T', title='Start date'),
    alt.Y('weighted_intensity:Q', title='Weighted cyber intensity'),
    tooltip=['name:N', 'start_date:T', 'weighted_intensity:Q', 'incident_type:N']
).properties(title='Severity of Incidents Targeting the US Over Time', width=600)

In [ ]:
by_type = us.assign(incident_type=us['incident_type'].str.split(';')).explode('incident_type')
by_type['incident_type'] = by_type['incident_type'].str.strip()

alt.Chart(by_type).mark_rect().encode(
    alt.X('impact_indicator_label:N', title='Impact level', sort=['Minor', 'Low', 'Medium', 'Not available']),
    alt.Y('incident_type:N', title='Incident type'),
    alt.Color('count():Q', title='Incidents', scale=alt.Scale(scheme='blues')),
    tooltip=['incident_type:N', 'impact_indicator_label:N', 'count():Q']
).properties(title='Which Incident Types Have the Biggest Impact?', width=350, height=200)

### Step 1.6: Summary and Next Steps



---
## Dataset 2: Ransomware.live Victims
*Investigated by Gurpreet Kaur*

### Step 2.1: Data Source

Mousqueton, J. *Ransomware.live* [Data set]. https://www.ransomware.live (API documentation: https://www.ransomware.live/apidocs). Accessed November 16, 2026.

Ransomware.live is a project by security researcher Julien Mousqueton that monitors the leak sites where ransomware groups name the organizations they have attacked.

### Step 2.2: Loading the Data

Ransomware.live has no single download file. Its public API returns the victims for one month at a time (`/victims/<year>/<month>`). We downloaded every month from 2021 to the download date once, pausing between requests because the API rate-limits, and saved the result as a fixed snapshot in `data/ransomware_live_raw.json`. This notebook reads that snapshot and does not call the API, so the numbers stay the same each time it runs.

In [ ]:
# load the saved Ransomware.live snapshot from our GitHub repository
RAW_URL = github_url + 'ransomware_live_raw.json'

with urllib.request.urlopen(RAW_URL, timeout=120) as resp:
    rows = json.loads(resp.read().decode('utf-8'))
print(f"Loaded snapshot: {len(rows):,} records")

rw_raw = pd.DataFrame(rows)

In [ ]:
rw_raw.head()

In [ ]:
print(f"Rows: {len(rw_raw):,}    Columns: {rw_raw.shape[1]}")
print("Columns:", list(rw_raw.columns))

nested = [c for c in rw_raw.columns if rw_raw[c].map(lambda v: isinstance(v, (dict, list))).any()]
print("\nNested columns:", nested)

rw_raw.info()

### Step 2.3: Describing the Data

#### What One Row Represents
Each row is one claim: a victim organization listed by a ransomware group on its leak site. The same organization can appear more than once, for example if it was claimed by two groups or re-posted. The raw snapshot has 33,698 rows and 15 columns (see the output above).

#### Columns Overview
The core fields are `victim` (the organization's name), `group` (the ransomware group that claimed it), `attackdate` (the date of the incident), `country` (2-letter ISO code, such as US or DE), and `activity` (the victim's sector). The API also returns nested fields (lists and dictionaries) that we leave out of the analysis table.

#### Challenges and Limitations
1. A leak site only shows victims that the group chose to name. Victims who paid quietly, and groups that do not run leak sites, are invisible. Counts are a lower bound, not all ransomware attacks.
2. Ransomware.live keeps adding groups and sites to its monitoring, so part of any upward trend may reflect better tracking rather than more attacks.
3. Attackdate may be an estimate or the posting date, and the month endpoint may be organized by when a victim was discovered. We test this below.
4. Country and sector are not always filled in.
5. The same victim can appear several times.
6. The numbers change daily, so we cache a snapshot and cite the access date.
7. Victim names are real organizations, many of them hospitals, schools or small businesses. We aggregate before plotting and do not put names in public charts.
8. The site has a Terms & Conditions page. We cite the source and use the data for coursework only.

### Step 2.4: Cleaning the Data

#### Selecting and Renaming Columns
We drop the nested columns and make all column names lowercase with no extra spaces.

In [ ]:
# drop nested columns (lists and dictionaries) and tidy the column names
rw = rw_raw.drop(columns=nested).copy()
rw.columns = rw.columns.str.strip().str.lower()
print(rw.columns.tolist())

#### Handling Missing Values
Placeholder text such as "Not Found" or "Unknown" is turned into real missing values so it is not counted as a real country or sector.

In [ ]:
# turn placeholder text into real missing values
placeholders = {"", "not found", "unknown", "n/a", "none"}
for c in rw.select_dtypes(include=["object", "string"]).columns:
    s = rw[c].astype("string").str.strip()
    rw[c] = s.mask(s.str.lower().isin(placeholders))

print("Share of missing values per column (%):")
print((rw.isna().mean() * 100).round(1).sort_values(ascending=False).head(10))

#### Transforming the Data
We parse the dates, remove duplicate claims and impossible dates, and add year and month columns.

In [ ]:
def to_date(series):
    '''Parse mixed date strings to timezone-free datetimes; unparseable values become NaT.'''
    try:
        parsed = pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    except (TypeError, ValueError):
        parsed = pd.to_datetime(series, errors="coerce", utc=True)
    return parsed.dt.tz_localize(None)


rw["attack_date"] = to_date(rw["attackdate"])
if "discovered" in rw.columns:
    rw["discovered_date"] = to_date(rw["discovered"])

n0 = len(rw)

dupe_key = [c for c in ["victim", "group", "attack_date"] if c in rw.columns]
rw = rw.drop_duplicates(subset=dupe_key)
print(f"Duplicate rows removed:      {n0 - len(rw):,}")

ok = rw["attack_date"].between(pd.Timestamp("2013-01-01"), pd.Timestamp.today())
print(f"Bad or missing dates removed: {(~ok).sum():,}")
rw = rw[ok].copy()

rw["year"] = rw["attack_date"].dt.year
rw["year_month"] = rw["attack_date"].dt.to_period("M").dt.to_timestamp()
print(f"Rows kept: {len(rw):,}  (from {n0:,})")
print(f"Attack dates run from {rw['attack_date'].min():%Y-%m-%d} to {rw['attack_date'].max():%Y-%m-%d}")

#### Recoding Values
We fix "UK" to the official code "GB", set invalid country codes to missing, add readable country names, and create one `sector` column.

In [ ]:
iso2_to_name = {c.alpha_2: c.name for c in pycountry.countries}
rw["country_code"] = rw["country"].str.upper().replace({"UK": "GB"})
unknown_codes = sorted(set(rw.loc[rw["country_code"].notna() & ~rw["country_code"].isin(iso2_to_name), "country_code"]))
print("Codes that are not valid ISO-2 (set to missing):", unknown_codes)
rw.loc[~rw["country_code"].isin(iso2_to_name), "country_code"] = pd.NA
rw["country_name"] = rw["country_code"].map(iso2_to_name)

sector_col = next((c for c in ["activity", "sector"] if c in rw.columns), None)
if sector_col:
    rw["sector"] = rw[sector_col]
    print(f"Sector column found: '{sector_col}' ({rw['sector'].nunique()} distinct values)")
else:
    print("No sector column in this download. Check rw_raw.columns or /sectorvictims/<sector>.")

print(f"\nRows with a country: {rw['country_code'].notna().mean():.1%}")
if sector_col:
    print(f"Rows with a sector:  {rw['sector'].notna().mean():.1%}")

#### Checking the Dates

In [ ]:
if "requested_month" in rw.columns:
    share = (rw["attack_date"].dt.strftime("%Y-%m") == rw["requested_month"]).mean()
    print(f"{share:.1%} of records have an attack date in the month they were requested under")

if "discovered_date" in rw.columns:
    lag = (rw["discovered_date"] - rw["attack_date"]).dt.days
    print("\nDays between attack date and discovery date:")
    print(lag.describe().round(1))

The month endpoint follows attackdate closely: 97.6% of records have an attack date in the month they were requested under, so grouping by attackdate is safe for our monthly and yearly counts. The gap between the attack date and the discovery date is short for most victims. Across the 33,694 records that have both dates, the median gap is 0 days and 75% of records are within 5 days. The average (20.1 days) is much higher than the median because of a few extreme values (standard deviation 87.0 days), with the largest gap being 1,456 days, a victim posted about four years after the attack. The minimum is -1,144 days, which means a victim was "discovered" about three years before its attack date. This is impossible and points to data errors or back-dated entries. Because at least a quarter of records have a gap of exactly 0 days, attackdate is often probably the same as the date the victim appeared on the leak site rather than the true date of the attack. We therefore keep attackdate as our time variable but treat our monthly trends as approximate, showing when victims were publicly named.

**Cleaning recap:** dropped nested columns and tidied column names, turned placeholder text into missing values, parsed dates, dropped duplicates and impossible dates, added year and month columns, converted country codes to names, and found the sector column. After cleaning, each row is one unique victim claim.

### Step 2.5: Exploratory Data Analysis

Three charts of the cleaned table rw, one for each question we can later compare with EuRepoC: when victims were posted, where they are, and which sectors they belong to. The same three questions are the ones the two datasets share (dates, countries and sectors or target categories).

Two choices apply to all three charts:
- The API returned no victims before April 2022, and the current month is still in progress. We therefore chart complete months only, from April 2022 up to last month.
- All charts use the Ransomware.live red that our group uses for this dataset.

In [ ]:
RW_COLOR = "#e45756"

first_month = pd.Timestamp("2022-04-01")
this_month = pd.Timestamp.today().to_period("M").to_timestamp()
rw_eda = rw[(rw["year_month"] >= first_month) & (rw["year_month"] < this_month)].copy()

print(f"{len(rw_eda):,} of {len(rw):,} victims fall in complete months "
      f"({rw_eda['year_month'].min():%b %Y} to {rw_eda['year_month'].max():%b %Y})")

**Chart 1. When were victims posted?**

In [ ]:
monthly = rw_eda.groupby("year_month").size().rename("victims").reset_index()
first, last = monthly["year_month"].min(), monthly["year_month"].max()

alt.Chart(monthly).mark_line(color=RW_COLOR, point=alt.OverlayMarkDef(color=RW_COLOR)).encode(
    x=alt.X("year_month:T", title="Month", axis=alt.Axis(format="%Y", tickCount="year")),
    y=alt.Y("victims:Q", title="Victims posted"),
    tooltip=[alt.Tooltip("year_month:T", title="Month", format="%B %Y"),
             alt.Tooltip("victims:Q", title="Victims", format=",")]
).properties(title=f"Ransomware Victims Posted per Month ({first:%b %Y} to {last:%b %Y})",
             width=650, height=300)

How many victims were posted each month, and is the number growing? That part of any growth can come from Ransomware.live monitoring more groups over time (challenge 2), and that attackdate is often the date the victim was posted.

**Chart 2. Where are the victims?**

In [ ]:
known = rw_eda.dropna(subset=["country_name"])
top_countries = (known["country_name"].value_counts().head(10)
                      .rename_axis("country").reset_index(name="victims"))

bars = alt.Chart(top_countries).mark_bar(color=RW_COLOR).encode(
    x=alt.X("victims:Q", title="Number of victims", axis=alt.Axis(tickCount=6),
            scale=alt.Scale(domainMax=top_countries["victims"].max() * 1.12)),
    y=alt.Y("country:N", sort="-x", title=None),
    tooltip=["country:N", alt.Tooltip("victims:Q", format=",")])
labels = bars.mark_text(align="left", dx=3).encode(text=alt.Text("victims:Q", format=","))

(bars + labels).properties(
    title=alt.TitleParams("Top 10 Countries by Ransomware Victims",
                          subtitle=f"Only the {len(known) / len(rw_eda):.0%} of victims that have a country"),
    width=500)

Only victims with a country are counted, and the title shows what share that is. These are raw counts, so large countries are expected to lead. The World Bank table (iso2, population_millions) will let us turn this into victims per million people.

**Chart 3. Which sectors are hit, and did the mix change?**

In [ ]:
sec = rw_eda.dropna(subset=["sector"])
sec_year = sec.groupby(["year", "sector"]).size().rename("victims").reset_index()
sec_year["share"] = sec_year["victims"] / sec_year.groupby("year")["victims"].transform("sum")
sector_order = sec["sector"].value_counts().index.tolist()
dark = (sec_year["share"].min() + sec_year["share"].max()) / 2   # cells above the midpoint get white text

cells_ = alt.Chart(sec_year).mark_rect().encode(
    x=alt.X("year:O", title="Year", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("sector:N", sort=sector_order, title=None),
    color=alt.Color("share:Q", scale=alt.Scale(scheme="reds"),
                    legend=alt.Legend(title="Share of year", format=".0%")),
    tooltip=["year:O", "sector:N", alt.Tooltip("victims:Q", format=","), alt.Tooltip("share:Q", format=".1%")])
numbers = alt.Chart(sec_year).mark_text(fontSize=10).encode(
    x="year:O", y=alt.Y("sector:N", sort=sector_order),
    text=alt.Text("share:Q", format=".0%"),
    color=alt.condition(f"datum.share > {dark}", alt.value("white"), alt.value("black")))

(cells_ + numbers).properties(
    title=alt.TitleParams("Share of Each Year's Victims by Sector",
                          subtitle=f"Only the {len(sec) / len(rw_eda):.0%} of victims that have a sector"),
    width=330, height=330)

For each year, what share of victims belongs to each sector? Shares (not counts) are used because 2022 (April to December) and 2026 (January to last month) are partial years, and the sector label is missing for a small part of the data. Sectors are sorted from most to least common overall.

### Step 2.6: Summary and Next Steps

Ransomware.live is a large, fast-growing list of criminal victims with a date, a country and a sector. It is the volume side of the story. EuRepoC is the opposite: a smaller list of *politically significant* incidents with rich detail about who attacked whom. Together they let us ask whether the two kinds of attacks follow the same geography and timeline.

**Patterns and relationships worth investigating**
1. **Trend over time.** Do monthly ransomware victims rise after 2020, and does the jump in EuRepoC incidents against the US (about 70 in 2022 to about 236 in 2023 in the EuRepoC chart above) coincide with it? We must keep in mind that part of the growth is better monitoring (challenge 2).
2. **Geography.** Which countries have the most victims, and does that change once we divide by population? Are those the same countries that appear most often as targets in EuRepoC?
3. **Do the two datasets move together?** A country-level correlation between EuRepoC incidents and ransomware victims, and a direct comparison of EuRepoC's own *Ransomware* incidents (about 210 against the US) with the far larger number of leak-site victims. The gap shows how little of ransomware reaches the "politically significant" list.
4. **Concentration among groups.** Are a handful of groups responsible for most victims, and how quickly do top groups appear and disappear?
5. **Sectors.** Which sectors does ransomware hit compared with EuRepoC's receiver categories (critical infrastructure, state institutions, corporate targets)?

It needs more data we can use from below:
- **Population** (World Bank, Dataset 3) to normalize country counts. A possible fourth source is the ITU Global Cybersecurity Index, to test whether countries with higher cybersecurity scores see fewer victims.
- A hand-made **crosswalk** between Ransomware.live's sector labels and EuRepoC's `receiver_category`, because the two use different vocabularies.

**Still to do.** Confirm the real column names after the first full download, decide how to treat the missing country and sector values, and agree with the group on the 2020-2024 window and on which EuRepoC incidents count as "state-linked" (the `initiator_category` column looks useful).

**Visualizations we are interested in:** a monthly trend with both datasets, a log-log country scatter plot, a country-by-year heatmap, a group-share area chart, a side-by-side sector comparison, and a world map of victims per million.

---
## Dataset 3: World Bank Country Indicators
*Investigated by Badamgarav Battushig*

### Step 3.1: Data Source

Source: World Bank (2026). *World Development Indicators*. Retrieved November 16, 2026, from https://data.worldbank.org

The World Bank publishes yearly statistics for almost every country in the world. We use three of them:

| Indicator | Code | Source page |
|---|---|---|
| Population, total | `SP.POP.TOTL` | https://data.worldbank.org/indicator/SP.POP.TOTL |
| GDP per capita (current US$) | `NY.GDP.PCAP.CD` | https://data.worldbank.org/indicator/NY.GDP.PCAP.CD |
| Individuals using the Internet (% of population) | `IT.NET.USER.ZS` | https://data.worldbank.org/indicator/IT.NET.USER.ZS |

The files were downloaded from the pages above and saved in our group GitHub repository: https://github.com/SU-DataViz-Group-Fall-2026/ransomware-analysis/tree/main/data

Data Choice: A large country like the United States will naturally have more attacks than a small one, simply because it has more people, companies, and computers. This dataset tells us how many people live in each country, how wealthy it is, and how many people use the internet. With it we can make fair comparisons, such as **attacks per million people**, and check whether richer or more connected countries are targeted more often.

### Step 3.2: Loading the Data

In [ ]:
# the 3 indicator files start with 4 rows of notes, so we skip them
population = pd.read_csv(github_url + 'wb_population.csv', skiprows=4)
gdp = pd.read_csv(github_url + 'wb_gdp_per_capita.csv', skiprows=4)
internet = pd.read_csv(github_url + 'wb_internet_users.csv', skiprows=4)

# country metadata (region and income group for each country)
metadata = pd.read_csv(github_url + 'wb_country_metadata.csv')

population.head()

In [ ]:
population.columns

In [ ]:
population.shape

In [ ]:
metadata.head()

In [ ]:
metadata.shape

### Step 3.3: Describing the Data

We loaded four CSV files: three indicator files and one metadata file.

#### What One Row Represents
- **Indicator files** (population, GDP per capita, internet users): each row is one country or region, and every year from 1960 to 2025 has its own column, so you read across a row to see how a country changed over time. Each file has 266 rows and 71 columns.
- **Metadata file:** each row is one country or region, with the world region it belongs to (e.g. North America) and its income group (low, lower middle, upper middle, or high income). It has 265 rows.

#### Columns Overview
`Country Name`, `Country Code` (a 3-letter code, e.g. USA), `Indicator Name`, `Indicator Code`, one column per year, and an empty column at the end caused by an extra comma in the file. The metadata file has `Country Code`, `Region`, `IncomeGroup`, `SpecialNotes`, and `TableName`.

#### Challenges and Limitations
1. Extra notes at the top. Each file has 4 lines of notes before the real table starts. We skip those lines.

2. Years are sideways. Each year has its own column (2000, 2001, 2002...). That makes charts hard, so we turn the table so all the years go in one "year" column.

3. Some rows are not countries. 48 rows are groups like "World" or "High income." If we keep them, they get counted like real countries. We can find them easily because they have no region listed, so remove them.

4. Some numbers are missing. 2025 is mostly empty, so only use 2000–2024. North Korea is one of the main attacker countries, but it has no GDP numbers and only 5 years of internet numbers.

5. Country names do not match. The World Bank writes countries one way (USA, Russian Federation). EuRepoC and Ransomware.live write them another way (US). We add the short 2-letter code to the World Bank data so all the files can be joined.

### Step 3.4: Cleaning the Data

#### Reshaping Years into One Column

In [ ]:
# helper: turn one indicator file from wide (one column per year) to long (one year column)
def to_long(data, value_name):
    long = data.melt(id_vars=['Country Name', 'Country Code'],
                     value_vars=[str(y) for y in range(2000, 2025)],
                     var_name='year', value_name=value_name)
    return long

# keep 2000-2024 (2025 is mostly empty) and melt each file
pop_long = to_long(population, 'population')
gdp_long = to_long(gdp, 'gdp_per_capita')
net_long = to_long(internet, 'internet_users_pct')

pop_long.head()

#### Combining the Indicators

In [ ]:
# combine the 3 indicators into one table: one row per country per year
wb = pop_long.merge(gdp_long, on=['Country Name', 'Country Code', 'year'])
wb = wb.merge(net_long, on=['Country Name', 'Country Code', 'year'])

# rename columns so they are easier to read
wb = wb.rename(columns={'Country Name': 'country', 'Country Code': 'iso3'})

wb.head()

#### Removing Non-Country Rows

In [ ]:
# keep only the columns we need from the metadata and rename them
meta = metadata[['Country Code', 'Region', 'IncomeGroup']]
meta = meta.rename(columns={'Country Code': 'iso3', 'Region': 'region', 'IncomeGroup': 'income_level'})

# remove regions like "World" and "High income" (they have no Region in the metadata)
real_countries = meta.dropna(subset=['region'])
print(len(meta), 'rows in metadata,', len(real_countries), 'real countries')

# keep only real countries and add region and income level
wb = wb.merge(real_countries, on='iso3', how='inner')
print(wb.shape)

#### Renaming and Recoding Values

In [ ]:
# rename income levels to shorter labels for charts
wb['income_level'] = wb['income_level'].replace({'Low income': 'Low',
                                                 'Lower middle income': 'Lower middle',
                                                 'Upper middle income': 'Upper middle',
                                                 'High income': 'High'})

# shorten long country names so they read better in charts
wb['country'] = wb['country'].replace({'Russian Federation': 'Russia',
                                       'Iran, Islamic Rep.': 'Iran',
                                       "Korea, Dem. People's Rep.": 'North Korea',
                                       'Korea, Rep.': 'South Korea',
                                       'Egypt, Arab Rep.': 'Egypt',
                                       'Turkiye': 'Turkey',
                                       'Viet Nam': 'Vietnam'})

# make year a whole number
wb['year'] = wb['year'].astype(int)

#### Handling Missing Values

In [ ]:
# check missing values
wb.isna().sum()

In [ ]:
# remove rows with no population (we can't compute "per million people" without it)
# we keep rows with missing GDP or internet use, so we don't lose the population data
wb = wb.dropna(subset=['population'])

#### Creating New Columns

In [ ]:
# population in millions is easier to read and needed for "attacks per million people"
wb['population_millions'] = wb['population'] / 1000000

# add a 2-letter country code so we can join with EuRepoC and Ransomware.live
# (Channel Islands has no 2-letter code, so it shows as "not found")
codes = list(wb['iso3'].unique())
iso2_codes = coco.convert(codes, to='ISO2')
wb['iso2'] = wb['iso3'].map(dict(zip(codes, iso2_codes)))

print(wb.shape)
wb.head()

**Cleaning recap:** skipped the note rows, reshaped the years into one column (2000-2024), combined the three indicators into one table, removed the 48 non-country groups, added region and income level, shortened long labels (for example "Russian Federation" became "Russia"), removed rows with no population, and added population in millions and a 2-letter country code. **After cleaning, each row is one country in one year.**

### Step 3.5: Exploratory Data Analysis

In [ ]:
# most recent year with good coverage
latest = wb[wb['year'] == 2024]

# top 10 countries by population
top_pop = latest.sort_values('population', ascending=False).head(10)

alt.Chart(top_pop).mark_bar().encode(
    alt.X('population_millions:Q', title='Population (millions)'),
    alt.Y('country:N', sort='-x', title='Country')
).properties(title='Most Populated Countries (2024)', width=450)

In [ ]:
# internet use over time for the main countries in our cyber story
story = ['United States', 'China', 'Russia', 'Iran', 'Ukraine', 'Germany', 'United Kingdom', 'Israel']
story_df = wb[wb['country'].isin(story)]

alt.Chart(story_df).mark_line(point=True).encode(
    alt.X('year:O', title='Year'),
    alt.Y('internet_users_pct:Q', title='Internet users (% of population)', scale=alt.Scale(domain=[0, 100])),
    alt.Color('country:N', title='Country')
).properties(title='Internet Use in Key Cyber Attack Countries', width=500)

In [ ]:
# how connected are countries around the world?
alt.Chart(latest).mark_bar().encode(
    alt.X('internet_users_pct:Q', bin=alt.Bin(step=10), title='Internet users (% of population)'),
    alt.Y('count():Q', title='Number of countries')
).properties(title='Internet Use Across Countries (2024)', width=450)

In [ ]:
# number of countries in each income level
income_order = ['Low', 'Lower middle', 'Upper middle', 'High']

alt.Chart(latest).mark_bar().encode(
    alt.X('income_level:N', sort=income_order, title='Income level'),
    alt.Y('count():Q', title='Number of countries')
).properties(title='Countries by World Bank Income Level', width=350)

In [ ]:
# average internet use by income level
by_income = latest.groupby('income_level')['internet_users_pct'].mean().reset_index()

alt.Chart(by_income).mark_bar().encode(
    alt.X('income_level:N', sort=income_order, title='Income level'),
    alt.Y('internet_users_pct:Q', title='Average internet users (% of population)', scale=alt.Scale(domain=[0, 100]))
).properties(title='Richer Countries Are More Connected (2024)', width=350)

In [ ]:
# do richer countries have more internet users? (scatter plot for EDA only)
alt.Chart(latest).mark_circle(opacity=0.6, size=50).encode(
    alt.X('gdp_per_capita:Q', scale=alt.Scale(type='log'), title='GDP per capita (US$, log scale)'),
    alt.Y('internet_users_pct:Q', title='Internet users (% of population)'),
    alt.Color('income_level:N', sort=income_order, title='Income level')
).properties(title='GDP per Capita vs. Internet Use (2024)', width=450)

### Step 3.6: Summary and Next Steps

What the data shows so far
- The cleaned table has 5,425 rows: 217 countries over 25 years (2000-2024).
- Countries differ hugely in size. In 2024, India had about 1,451 million people and China about 1,409 million, more than four times the United States (about 340 million). Simply counting attacks would make big countries look the most targeted, so we will also look at attacks per million people.
- Internet use grew quickly in all the key countries, from under 50% in 2000 to over 80% by 2024, which matches the rise in cyber incidents after 2010.
- **Richer countries are more connected.** In 2024, about 92% of people in high-income countries used the internet, compared with only 23% in low-income countries.

Questions worth exploring
- Which countries are hit hardest per person once we account for population?
- Do richer countries have more ransomware victims, since ransomware gangs are after money?
- Does internet use or wealth better explain attacks?

Does it need more data?
Yes. This dataset has no cyber attack data on its own. It adds context to EuRepoC and Ransomware.live, and all three can be matched using the 2-letter country code and year.

Visualizations we would like to make
1. A world map of cyber incidents per million people, next to a map of raw counts.
2. A bar chart of ransomware victims per million people by income group.
3. A ranked bar chart of the top 15 countries by attacks per million people.
4. Side-by-side line charts comparing internet growth with yearly cyber incidents.

---
## Dataset 4: The Ransomware Decade
*Investigated by Jack Lichwa*

### Step 4.1: Data Source

Ransomware Decade dataset, Zenodo record 15571866 (`incidents.csv` and `schema.json`). License: CC BY-NC-ND 4.0. Original record: https://zenodo.org/records/15571866

It records one ransomware incident per row. Most detail columns come from extracted news articles (see the `sources` column in the schema).

### Step 4.2: Loading the Data

In [ ]:
import ast
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", 30)

RD_DATA_PATH = "../data/ransomware_decade_incidents.csv"
rd_raw = pd.read_csv(RD_DATA_PATH)
print(rd_raw.shape)
# Victim and join key, then the fields Ransomware Decade adds beyond Ransomware.live

rd_raw.head()

In [ ]:
rd_raw[["victim", "incident-date", "attacker", "data-exfiltration",
        "impact.category.Operational Impact", "affected-data.category.PII"]].head()

### Step 4.3: Describing the Data

#### What One Row Represents

One row is one ransomware incident against one victim organization. `uuid` is unique for every row.

#### Columns Overview

The file has 4,070 rows and 143 columns. The overview table below gives the key counts and the date range. Of the 143 columns, 101 are True/False fields.

#### Challenges and Limitations

- About half of the 2020 to 2024 incidents have no attacker recorded (see the check table in Step 4.4).
- The True/False fields have no blank values, so a False may mean "not reported" rather than "did not happen."
- The file has no country column, so it cannot be compared with the country-based datasets 1 to 3 directly.
- Only about 31.5% of rows have a Ransomware.live link (overview table).

In [ ]:
# Full-file rd_overview: rows, columns, and how complete the key fields are
rd_dates = pd.to_datetime(rd_raw["incident-date"], errors="coerce")
rd_overview = pd.DataFrame(
    [
        ("Rows (one row = one incident)", len(rd_raw)),
        ("Columns", rd_raw.shape[1]),
        ("Unique uuid values", rd_raw["uuid"].nunique()),
        ("First incident date", rd_dates.min().date()),
        ("Last incident date", rd_dates.max().date()),
        ("Boolean columns", len(rd_raw.select_dtypes("bool").columns)),
        ("Blank values in boolean columns", int(rd_raw.select_dtypes("bool").isna().sum().sum())),
        ("Share with a Ransomware.live link", round(rd_raw["ransomware.live-url"].notna().mean(), 3)),
    ],
    columns=["measure", "value"],
).set_index("measure")
rd_overview

### Step 4.4: Cleaning the Data

#### Selecting and Renaming Columns

In [ ]:
# Keep only the fields the analysis uses
rd_df = rd_raw[["incident-date", "attacker", "data-exfiltration",
               "impact.category.Operational Impact", "affected-data.category.PII"]].copy()

rd_df["incident_year"] = pd.to_datetime(rd_df["incident-date"], errors="coerce").dt.year

# Attackers: parse the list text, label empty lists "Unattributed" (kept, not dropped)
rd_df["rd_attacker_list"] = rd_df["attacker"].fillna("[]").apply(lambda s: ast.literal_eval(s) if s.strip() else [])
rd_df["attacker_primary"] = rd_df["rd_attacker_list"].apply(lambda xs: xs[0] if xs else "Unattributed")

# True/False fields as readable text
rd_df["exfiltration"] = rd_df["data-exfiltration"].map({True: "Yes", False: "No"})
rd_df["operational_impact"] = rd_df["impact.category.Operational Impact"].map({True: "Yes", False: "No"})
rd_df["affected_pii"] = rd_df["affected-data.category.PII"].map({True: "Yes", False: "No"})

# Working table: 2020 to 2024 rows and the focus fields only
focus_cols = ["incident_year", "attacker_primary", "exfiltration", "operational_impact", "affected_pii"]
rd_focus = rd_df.loc[rd_df["incident_year"].between(2020, 2024), focus_cols].reset_index(drop=True)

rd_clean_summary = pd.DataFrame({
    "filled": rd_focus.notna().sum(),
    "missing": rd_focus.isna().sum(),
})
rd_clean_summary

#### Handling Missing Values

In [ ]:
# Data-quality check for the focus columns, 2020 to 2024 only, from the full file
rd_map = {
    "incident-date": "incident_year",
    "attacker": "attacker_primary",
    "data-exfiltration": "exfiltration",
    "impact.category.Operational Impact": "operational_impact",
    "affected-data.category.PII": "affected_pii",
}
rd_year = pd.to_datetime(rd_raw["incident-date"], errors="coerce").dt.year
rd_window = rd_raw[rd_year.between(2020, 2024)]
rd_check = pd.DataFrame({
    "cleaned column": list(rd_map.values()),
    "rows in 2020 to 2024": len(rd_window),
    "blank values": [int(rd_window[c].isna().sum()) for c in rd_map],
    "no attacker or False": [
        int((rd_window[c] == "[]").sum()) if c == "attacker"
        else int((rd_window[c] == False).sum()) if c.startswith(("data-", "impact", "affected"))
        else 0
        for c in rd_map
    ],
}, index=pd.Index(list(rd_map.keys()), name="source column"))
rd_check

#### Recoding Values

In [ ]:
rd_df[["attacker_primary", "exfiltration", "operational_impact", "affected_pii"]].head()

#### Transforming the Data

In [ ]:
rd_clean_summary

**Cleaning recap:** the cleaning keeps only the incident date, attacker, exfiltration, operational impact, and personal data affected, and drops the other 138 columns. The year comes from the incident date, and the three True/False fields become Yes/No text so a missing value is never read as False. The attacker list is parsed and the first name becomes the primary attacker, with empty lists labeled "Unattributed" rather than dropped. The working table then keeps the 3,359 incidents from 2020 to 2024. Every focus column is filled for all 3,359 rows. The file has no blank True/False values, so a False could mean "not reported" rather than "did not happen." Of the 3,359 incidents, 1,619 (48%) have no attacker, 1,773 are False for exfiltration, 1,658 for operational impact, and 2,119 for personal data affected.

### Step 4.5: Exploratory Data Analysis

#### Incidents per year

In [ ]:
rd_per_year = rd_focus["incident_year"].value_counts().sort_index().rename("incidents").to_frame()
rd_per_year

In [ ]:
rd_year_chart = (
    alt.Chart(rd_per_year.reset_index())
    .mark_bar(color="#4e79a7")
    .encode(
        x=alt.X("incident_year:O", title="Year"),
        y=alt.Y("incidents:Q", title="Incidents"),
        tooltip=["incident_year:O", "incidents:Q"],
    )
    .properties(width=480, height=280, title="Incidents per year, 2020 to 2024")
)
rd_year_chart

**Chart justification:** bar length on a shared baseline shows the count for each year, so the year-to-year change is easy to compare. One hue is used, since the year is the only grouping.

#### Top attributed ransomware groups

In [ ]:
rd_focus["attacker_primary"].value_counts().head(12).rename("incidents").to_frame()

In [ ]:
rd_groups = rd_focus["attacker_primary"].value_counts().head(12).rename("incidents").reset_index()
rd_groups.columns = ["attacker_primary", "incidents"]
rd_group_chart = (
    alt.Chart(rd_groups)
    .mark_bar()
    .encode(
        y=alt.Y("attacker_primary:N", sort="-x", title="Attacker group"),
        x=alt.X("incidents:Q", title="Incidents"),
        color=alt.condition(
            alt.datum.attacker_primary == "Unattributed",
            alt.value("#999999"),
            alt.value("#4e79a7"),
        ),
        tooltip=["attacker_primary:N", "incidents:Q"],
    )
    .properties(width=480, height=300, title="Top attacker rd_groups, including Unattributed")
)
rd_group_chart

**Chart justification:** horizontal bars sorted by count keep the group names readable. Gray marks "Unattributed", the only no-data category, so it is visibly different from the named groups in blue.

#### Data exfiltration

In [ ]:
rd_exfil = rd_focus["exfiltration"].value_counts().rename("incidents").to_frame()
rd_exfil["share"] = (rd_exfil["incidents"] / len(rd_focus)).round(3)
rd_exfil

In [ ]:
rd_exfil_chart = (
    alt.Chart(rd_exfil.reset_index())
    .mark_bar()
    .encode(
        x=alt.X("exfiltration:N", title="Data exfiltrated", sort=["Yes", "No"]),
        y=alt.Y("share:Q", title="Share of incidents", axis=alt.Axis(format="%")),
        color=alt.Color("exfiltration:N", title="Data exfiltrated", scale=alt.Scale(range=["#e15759", "#4e79a7"])),
        tooltip=["exfiltration:N", "incidents:Q", alt.Tooltip("share:Q", format=".1%")],
    )
    .properties(width=320, height=260, title="Data exfiltration, 2020 to 2024")
)
rd_exfil_chart

**Chart justification:** color separates Yes (red) from No (blue), and bar height shows the share, matching the table above.

#### Operational impact

In [ ]:
rd_impact = rd_focus["operational_impact"].value_counts().rename("incidents").to_frame()
rd_impact["share"] = (rd_impact["incidents"] / len(rd_focus)).round(3)
rd_impact

In [ ]:
rd_impact_chart = (
    alt.Chart(rd_impact.reset_index())
    .mark_bar()
    .encode(
        x=alt.X("operational_impact:N", title="Operational impact", sort=["Yes", "No"]),
        y=alt.Y("share:Q", title="Share of incidents", axis=alt.Axis(format="%")),
        color=alt.Color("operational_impact:N", title="Operational impact", scale=alt.Scale(range=["#e15759", "#4e79a7"])),
        tooltip=["operational_impact:N", "incidents:Q", alt.Tooltip("share:Q", format=".1%")],
    )
    .properties(width=320, height=260, title="Operational impact, 2020 to 2024")
)
rd_impact_chart

**Chart justification:** color separates Yes (red) from No (blue), and bar height shows the share, matching the table in 3d.

#### Personal data affected

In [ ]:
rd_pii = rd_focus["affected_pii"].value_counts().rename("incidents").to_frame()
rd_pii["share"] = (rd_pii["incidents"] / len(rd_focus)).round(3)
rd_pii

In [ ]:
rd_pii_chart = (
    alt.Chart(rd_pii.reset_index())
    .mark_bar()
    .encode(
        x=alt.X("affected_pii:N", title="Personal data affected", sort=["Yes", "No"]),
        y=alt.Y("share:Q", title="Share of incidents", axis=alt.Axis(format="%")),
        color=alt.Color("affected_pii:N", title="Personal data affected", scale=alt.Scale(range=["#e15759", "#4e79a7"])),
        tooltip=["affected_pii:N", "incidents:Q", alt.Tooltip("share:Q", format=".1%")],
    )
    .properties(width=320, height=260, title="Personal data affected, 2020 to 2024")
)
rd_pii_chart

**Chart justification:** color separates Yes (red) from No (blue), and bar height shows the share.

### Step 4.6: Summary

#### Patterns Worth Investigating

- About half of the 2020 to 2024 incidents have no attacker recorded, so group comparisons cover only the attributed half.
- Exfiltration (47% Yes) and operational impact (51% Yes) are close to even, so whether they occur together is worth checking.
- Personal data affected is less common (37% Yes).

#### Does It Need More Data?

Yes, to say more about the groups and countries. The Decade data has no country column, so comparing it with the country-based datasets is left as a description rather than a direct merge. Its 2020 to 2024 window is supplementary to the other datasets.

#### Visualizations We Are Interested In

Visualization 6 below.

---
## Combining the Datasets

### Matching by Country Code and Year
Datasets 1-3 can be matched on a **2-letter country code** and **year** (Dataset 4, the Decade data, has no country column, so it is described against the others rather than matched directly):

| Dataset | Country column | Year from |
|---|---|---|
| EuRepoC | `receiver_country_alpha_2_code` (victim country) | `start_date` |
| Ransomware.live | `country_code` | `attack_date` |
| World Bank | `iso2` | `year` |
| Dataset 4 | * | * |

The same attack can appear in both EuRepoC and Ransomware.live, so we **compare** their counts side by side instead of adding them together.

---
## Visualization Brainstorm

As a group, we brainstormed at least 5 visualizations that could be made with our combined data. For each one, we give the question it answers, the variables and calculations, the encodings, the chart type, and the design considerations.

### Visualization 1: Which attacker countries target which sectors?

#### Variables and Calculations
`initiator_country` and `receiver_category` (EuRepoC). Split both multi-value columns and count each sector once per incident. Keep the top attacker countries and the main sectors, and exclude unattributed incidents.

#### Encodings
x = target sector, y = attacker country, color = number of incidents.

#### Chart Type
Heatmap.

#### Design Considerations
Use a sequential single-hue scale, since counts go from low to high. Sort countries by total incidents. Add the count as text in each cell. Consider showing the % of each country's incidents instead of raw counts, so countries with fewer incidents can still be compared.

### Visualization 2: Did monthly ransomware victims grow because each group posted more, or because more groups were active?

#### Variables and Calculations
`year_month`, `group`, `victim` (Ransomware.live). Compute victims per month and active groups per month (number of unique groups). Index both to their first six-month average = 100. Use complete months only.

#### Encodings
x = month, y = index, color = measure.

#### Chart Type
Indexed two-line chart, with an optional small panel for victims per active group.

#### Design Considerations
Red for victims (the Ransomware.live color) and grey for groups. Label the lines directly instead of using a legend, and draw a baseline at 100. Add a caption: "active group = posted at least one victim that month."

#### What the Data Shows So Far
Monthly victims average about 270 in 2022 and about 855 in 2026, and active groups go from about 25 to about 70. Victims per active group stay between 9.5 and 12.3 in every year, so growth is mostly more groups posting.

### Visualization 3: How long do the biggest ransomware groups stay active, and who takes over when one fades?

#### Variables and Calculations
`group`, `year_month` (Ransomware.live). Keep the top 15 groups by total victims, count victims per group-month, and sort groups by first month seen. Decide whether to merge lockbit2 and lockbit3.

#### Encodings
x = month, y = group, color = victims that month.

#### Chart Type
Activity-timeline heatmap. A yearly rank bump chart is the alternative.

#### Design Considerations
A single-hue red scale with a square-root transform, group names on the left, and year ticks only. Caption: "10 of 304 groups account for 43% of victims."

#### What the Data Shows So Far
The leader changes, from lockbit3 in 2022-23 to ransomhub in 2024 to qilin in 2025-26.

### Visualization 4: Do countries with more EuRepoC incidents also have more ransomware victims, and how much ransomware does EuRepoC capture?

#### Variables and Calculations
Country counts for 2024 only. For EuRepoC, split `receiver_country_alpha_2_code` on ";", drop repeated (incident, country) pairs, and flag `is_ransomware` (an `incident_type` value equal to "Ransomware"). For Ransomware.live, use `country_code`.

#### Encodings
x = EuRepoC incidents (log), y = ransomware victims (log), color = share of EuRepoC incidents tagged ransomware.

#### Chart Type
Log-log scatter plot with a light diagonal reference line.

#### Design Considerations
Label the top 8 countries, put the rank correlation in the subtitle, and annotate the two totals. Use 2024 only, because EuRepoC ends in December 2024 and Ransomware.live's country field is only reliable from January 2024.

#### What the Data Shows So Far
EuRepoC has 700 incidents in 2024, of which 220 are tagged ransomware, against 5,877 leak-site victims with a country. 87 countries appear in both. The rank correlation is 0.65, or 0.56 without EuRepoC's ransomware-tagged incidents.

### Visualization 5: Which countries are hit hardest by cyber incidents once we account for population?

#### Variables and Calculations
EuRepoC `receiver_country_alpha_2_code` and World Bank `iso2`, `population_millions` (2024). Split the multi-value country codes on ";", drop repeated (incident, country) pairs, and keep only real 2-letter country codes (removes regions like EUROPE). Count incidents per country, join with the World Bank data on the 2-letter code, and compute incidents per million people. Keep only countries with at least 1 million people, so tiny countries don't dominate.

#### Encodings
Geographic shape = country, color = incidents per million people, tooltip = country, incident count, and incidents per million.

#### Chart Type
World choropleth map, shown next to a map of raw incident counts so readers can compare the two.

#### Design Considerations
Use an equal-area projection (equalEarth) and the same sequential single-hue color scheme on both maps. Show countries with no data in light gray. Add a subtitle explaining that values are per million people.

#### What the Data Shows So Far
In raw counts the US leads with 871 incidents, but per person the top countries are Israel (14.9 incidents per million people), Estonia (10.2), and Bahrain (6.9).

### Visualization 6: Which attacker groups more often steal data?

#### Variables and Calculations
`attacker_primary` (top 10 groups by incidents, plus "Unattributed") and `exfiltration`. For each group: incidents = row count, Yes count, share = Yes ÷ incidents. Filter to 2020 to 2024.

#### Encodings
y = group (nominal), sorted by share; x = share (quantitative); dot size = incidents (quantitative); color = blue, gray for "Unattributed".

#### Chart Type
Dot plot.

#### Design Considerations
x axis from 0 to 100%, ticks every 10%. Dot size legend at the smallest, median, and largest incident count. Each dot carries its percent label. Title "Exfiltration share by top attacker groups, 2020 to 2024". About 25 px per group in height, 480 wide.

---
## Publishing: GitHub Repository and GitHub Pages Website

Instructions for creating a GitHub repository and GitHub Pages website.

1. Sign in to GitHub or create an account.
2. Make a new repository by clicking the "New" button on the GitHub homepage. Title it something meaningful, like "DataViz" or similar.
3. Set the visibility to Public, toggle on to include a README, and optionally choose a license such as MIT.
4. Click "Create repository".

### Exporting the notebook to HTML and publishing it with GitHub Pages

5. Export the notebook as HTML. From the folder that contains the notebook, run:
   `jupyter nbconvert --to html "HW8_Group_Project-EDA.ipynb"`
   This writes `HW8_Group_Project-EDA.html` with the outputs (tables and charts) included.
6. Rename the file to `index.html`. GitHub Pages serves `index.html` as the home page.
7. In the repository, click "Add file" then "Upload files", add `index.html`, and commit it to the main branch.
8. Open the repository's "Settings" tab, then "Pages". Under "Build and deployment", set Source to "Deploy from a branch", Branch to "main", and folder to "/ (root)". Click "Save".
9. Wait a minute or two, then reload the Pages settings. The site address is `https://<your-username>.github.io/<repository-name>/`. Submit that link.

Only the notebook source is committed to the repository. The HTML file is the published copy with outputs, so it is the only file that carries outputs.

---
## References

1. European Repository of Cyber Incidents (EuRepoC). (2025). *Global Dataset of Cyber Incidents (Version 1.3)* [Data set]. Zenodo. https://zenodo.org/records/14965395
2. Mousqueton, J. *Ransomware.live* [Data set]. Retrieved November 16, 2026, from https://www.ransomware.live
3. World Bank. (2026). *World Development Indicators: Population, total (SP.POP.TOTL); GDP per capita, current US$ (NY.GDP.PCAP.CD); Individuals using the Internet, % of population (IT.NET.USER.ZS)* [Data set]. Retrieved November 16, 2026, from https://data.worldbank.org
4. Ransomware Decade. (2025). *The Ransomware Decade: incidents.csv and schema.json* [Data set]. Zenodo, record 15571866. https://zenodo.org/records/15571866 (CC BY-NC-ND 4.0)